# Classifier Chains & Ensemble of Classifier Chains

**Dataset:** GoodScents + Leffingwell  
**Preprocessing:** identical to baseline.ipynb (same split, same scaling, same correlation filter)

We compare two chain-based strategies against Binary Relevance:
- **CC** — single Classifier Chain (fixed random label order)
- **ECC** — Ensemble of Classifier Chains (10 chains, different random orderings, majority vote)

## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from skmultilearn.model_selection import iterative_train_test_split

## Load & Preprocess Data

Identical pipeline to baseline.ipynb.

In [2]:
df = pd.read_csv('goodscents_jadbio_ready.csv', sep=';')

LABEL_COLS = [
    'floral', 'fruity', 'sweet', 'woody', 'green', 'spicy',
    'animal_musk', 'earthy', 'citrus', 'chemical', 'gourmand', 'powdery_amber'
]

fp_cols      = [c for c in df.columns if c.startswith('MACCS_') or c.startswith('morgan_')]
mordred_cols = [c for c in df.columns if c not in LABEL_COLS + ['SMILES'] + fp_cols]

df_clean = df.dropna().reset_index(drop=True)

X_fp      = df_clean[fp_cols].values.astype(float)
X_mordred = df_clean[mordred_cols].values.astype(float)
y         = df_clean[LABEL_COLS].values

# Zero-variance filter
vt_fp = VarianceThreshold(threshold=0)
X_fp  = vt_fp.fit_transform(X_fp)

vt_mordred = VarianceThreshold(threshold=0)
X_mordred  = vt_mordred.fit_transform(X_mordred)

# Train/test split (iterative stratified)
X_combined = np.hstack([X_fp, X_mordred])
n_fp = X_fp.shape[1]

X_train_comb, y_train, X_test_comb, y_test = iterative_train_test_split(
    X_combined, y, test_size=0.2
)

X_fp_train    = X_train_comb[:, :n_fp]
X_mordred_train = X_train_comb[:, n_fp:]
X_fp_test     = X_test_comb[:, :n_fp]
X_mordred_test  = X_test_comb[:, n_fp:]

# Scale Mordred (fit on train only)
scaler = StandardScaler()
X_mordred_train = scaler.fit_transform(X_mordred_train)
X_mordred_test  = scaler.transform(X_mordred_test)

# Correlation filter on Mordred (train only)
corr_matrix    = np.abs(np.corrcoef(X_mordred_train.T))
upper_triangle = np.triu(corr_matrix, k=1)
cols_to_drop   = set()
for r, c in zip(*np.where(upper_triangle > 0.95)):
    if c not in cols_to_drop:
        cols_to_drop.add(c)
cols_to_keep    = [i for i in range(X_mordred_train.shape[1]) if i not in cols_to_drop]
X_mordred_train = X_mordred_train[:, cols_to_keep]
X_mordred_test  = X_mordred_test[:, cols_to_keep]

X_train = np.hstack([X_fp_train, X_mordred_train])
X_test  = np.hstack([X_fp_test,  X_mordred_test])

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}')
print(f'y_test  : {y_test.shape}')

X_train : (3883, 993)
X_test  : (1093, 993)
y_train : (3883, 12)
y_test  : (1093, 12)


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Shared Evaluation Helper

In [3]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, recall_score, precision_score,
    hamming_loss, jaccard_score, confusion_matrix
)

EPS = 1e-8

# ── Co-occurrence helpers ──────────────────────────────────────────────────────

def _cooc_matrix(Y):
    """Label co-occurrence matrix: entry [i,j] = # samples with both label i and j."""
    return (Y.T @ Y).astype(float)

def _label_cooc_consistency(y_train, y_pred):
    """
    LCC = mean |C/(Cmax+ε) − Ĉ/(Ĉmax+ε)|
    C   = train co-occurrence matrix (reference)
    Ĉ   = predicted co-occurrence matrix
    Lower is better (0 = perfect structural match).
    """
    C     = _cooc_matrix(y_train)
    C_hat = _cooc_matrix(y_pred)
    C_norm     = C     / (C.max()     + EPS)
    C_hat_norm = C_hat / (C_hat.max() + EPS)
    return float(np.mean(np.abs(C_norm - C_hat_norm)))

# ── Main evaluation function ───────────────────────────────────────────────────

def evaluate_12(name, y_tr, y_te, y_pred, y_prob, label_cols):
    """
    Returns a dict of 13 scalar metrics aggregated over all 12 labels.

    Parameters
    ----------
    name       : model name string
    y_tr       : training labels  (n_train, n_labels)  — for LCC reference
    y_te       : test ground truth (n_test,  n_labels)
    y_pred     : binary predictions (n_test, n_labels)
    y_prob     : probability scores  (n_test, n_labels)
    label_cols : list of label names (length n_labels)

    Metric directions (all higher = better except where noted)
    -----------------------------------------------------------
    roc_auc_12               ↑
    pr_auc_12                ↑
    f1_macro_12              ↑  macro F1 over labels
    instance_f1_12           ↑  sample-averaged F1
    balanced_accuracy_12     ↑  macro balanced accuracy
    matched_accuracy_12      ↑  exact match ratio
    sensitivity_macro_12     ↑  macro recall / TPR
    specificity_macro_12     ↑  macro TNR
    precision_macro_12       ↑  macro precision
    recall_macro_12          ↑  == sensitivity_macro_12
    hamming_loss_12          ↓  lower is better
    jaccard_12               ↑  macro Jaccard
    label_cooc_consistency_12↓  lower is better (divergence)
    """
    n_labels = len(label_cols)
    bal_accs, sens_list, spec_list, prec_list, roc_aucs, pr_aucs, f1s =         [], [], [], [], [], [], []

    for i in range(n_labels):
        yt, yp, ypr = y_te[:, i], y_pred[:, i], y_prob[:, i]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

        bal_accs.append(balanced_accuracy_score(yt, yp))
        sens_list.append(recall_score(yt, yp, zero_division=0))
        spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))
        prec_list.append(precision_score(yt, yp, zero_division=0))
        f1s.append(f1_score(yt, yp, zero_division=0))

        try:    roc_aucs.append(roc_auc_score(yt, ypr))
        except: roc_aucs.append(float('nan'))
        try:    pr_aucs.append(average_precision_score(yt, ypr))
        except: pr_aucs.append(float('nan'))

    instance_f1 = f1_score(y_te, y_pred, average='samples', zero_division=0)
    matched_acc = float(np.mean(np.all(y_te == y_pred, axis=1)))
    h_loss      = hamming_loss(y_te, y_pred)
    jaccard     = jaccard_score(y_te, y_pred, average='macro', zero_division=0)
    lcc         = _label_cooc_consistency(y_tr, y_pred)

    return {
        'model'                    : name,
        'roc_auc_12'               : round(float(np.nanmean(roc_aucs)),  4),
        'pr_auc_12'                : round(float(np.nanmean(pr_aucs)),   4),
        'f1_macro_12'              : round(float(np.mean(f1s)),          4),
        'instance_f1_12'           : round(instance_f1,                  4),
        'balanced_accuracy_12'     : round(float(np.mean(bal_accs)),     4),
        'matched_accuracy_12'      : round(matched_acc,                  4),
        'sensitivity_macro_12'     : round(float(np.nanmean(sens_list)), 4),
        'specificity_macro_12'     : round(float(np.nanmean(spec_list)), 4),
        'precision_macro_12'       : round(float(np.nanmean(prec_list)), 4),
        'recall_macro_12'          : round(float(np.nanmean(sens_list)), 4),
        'hamming_loss_12'          : round(h_loss,                       4),
        'jaccard_12'               : round(jaccard,                      4),
        'label_cooc_consistency_12': round(lcc,                          4),
    }

# ── Pretty printer ─────────────────────────────────────────────────────────────

_LOWER_BETTER = {'hamming_loss_12', 'label_cooc_consistency_12'}

def print_results_12(m):
    print(f"\n=== {m['model']} ===")
    col_w = 32
    for k, v in m.items():
        if k == 'model':
            continue
        direction = '↓' if k in _LOWER_BETTER else '↑'
        print(f"  {k:<{col_w}} {v:.4f}  {direction}")


# Model: Classifier Chain + Random Forest

We use RF as the base classifier (best performer in the baseline comparison).

The label order is randomly shuffled — CC is sensitive to ordering, so we fix a seed for reproducibility.
The chain trains 12 classifiers sequentially:
- Classifier 1: predicts label_order[0] from X
- Classifier 2: predicts label_order[1] from X + predicted label_order[0]
- ...
- Classifier 12: predicts label_order[11] from X + all 11 previous predictions

In [4]:
from sklearn.multioutput import ClassifierChain
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

scorer = make_scorer(f1_score, average='macro')

# ClassifierChain from sklearn wraps any base estimator
# order='random' + random_state fixes the label permutation
cc_rf = ClassifierChain(
    base_estimator=RandomForestClassifier(
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    order='random',
    random_state=42
)

# Note: GridSearchCV param prefix is 'base_estimator__' for ClassifierChain
gs_cc = GridSearchCV(
    cc_rf,
    {'base_estimator__n_estimators': [100, 300],
     'base_estimator__max_features': ['sqrt', 'log2']},
    scoring=scorer, cv=3, n_jobs=-1, verbose=1
)

print('Fitting CC + RF...')
gs_cc.fit(X_train, y_train)
print(f'Best params : {gs_cc.best_params_}  |  CV macro-F1 : {gs_cc.best_score_:.3f}')

best_cc   = gs_cc.best_estimator_
y_pred_cc = best_cc.predict(X_test)
y_prob_cc = best_cc.predict_proba(X_test)

# ClassifierChain returns predictions in chain order — reorder back to LABEL_COLS order
y_pred_cc = y_pred_cc[:, np.argsort(best_cc.order_)]
y_prob_cc = y_prob_cc[:, np.argsort(best_cc.order_)]

metrics_cc = evaluate_12('CC_RF', y_train, y_test, y_pred_cc, y_prob_cc, LABEL_COLS)
print_results_12(metrics_cc)


Fitting CC + RF...


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


Fitting 3 folds for each of 4 candidates, totalling 12 fits


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


Best params : {'base_estimator__max_features': 'sqrt', 'base_estimator__n_estimators': 300}  |  CV macro-F1 : 0.477


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)



=== CC_RF ===
  roc_auc_12                       0.4879  ↑
  pr_auc_12                        0.2601  ↑
  f1_macro_12                      0.1887  ↑
  instance_f1_12                   0.1697  ↑
  balanced_accuracy_12             0.5025  ↑
  matched_accuracy_12              0.0037  ↑
  sensitivity_macro_12             0.2145  ↑
  specificity_macro_12             0.7904  ↑
  precision_macro_12               0.2509  ↑
  recall_macro_12                  0.2145  ↑
  hamming_loss_12                  0.3597  ↓
  jaccard_12                       0.1096  ↑
  label_cooc_consistency_12        0.1642  ↓


# Model: Ensemble of Classifier Chains + Random Forest

ECC trains N chains, each with a **different random label ordering**.
Final prediction for each label = **majority vote** across the N chains.
Probability = **average predicted probability** across chains.

This reduces the sensitivity to any single ordering and generally outperforms a single CC.
We use N=10 chains — a standard choice from Read et al. (2011).

In [5]:
N_CHAINS = 10

# Use the best RF params found above
best_n_est    = gs_cc.best_params_['base_estimator__n_estimators']
best_max_feat = gs_cc.best_params_['base_estimator__max_features']

chains = [
    ClassifierChain(
        base_estimator=RandomForestClassifier(
            n_estimators=best_n_est,
            max_features=best_max_feat,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ),
        order='random',
        random_state=seed
    )
    for seed in range(N_CHAINS)
]

print(f'Training {N_CHAINS} chains...')
for i, chain in enumerate(chains):
    chain.fit(X_train, y_train)
    print(f'  Chain {i+1}/{N_CHAINS} done')

# Collect predictions from each chain, reordered back to LABEL_COLS
all_preds = np.array([
    chain.predict(X_test)[:, np.argsort(chain.order_)]
    for chain in chains
])  # shape: (N_CHAINS, n_samples, n_labels)

all_probs = np.array([
    chain.predict_proba(X_test)[:, np.argsort(chain.order_)]
    for chain in chains
])  # shape: (N_CHAINS, n_samples, n_labels)

# Majority vote for labels, mean for probabilities
y_pred_ecc = (all_preds.mean(axis=0) >= 0.5).astype(int)
y_prob_ecc = all_probs.mean(axis=0)

metrics_ecc = evaluate_12('ECC_RF', y_train, y_test, y_pred_ecc, y_prob_ecc, LABEL_COLS)
print_results_12(metrics_ecc)


Training 10 chains...


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


  Chain 1/10 done


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


  Chain 2/10 done


...

  Chain 9/10 done


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)


  Chain 10/10 done


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\multioutput.py:689: FutureWarning: `base_estimator` as an argument was deprecated in 1.7 and will be removed in 1.9. Use `estimator` instead.
  warnings.warn(warning_msg, FutureWarning)
C:\Users\h


=== ECC_RF ===
  roc_auc_12                       0.5453  ↑
  pr_auc_12                        0.2927  ↑
  f1_macro_12                      0.1500  ↑
  instance_f1_12                   0.0999  ↑
  balanced_accuracy_12             0.5150  ↑
  matched_accuracy_12              0.0018  ↑
  sensitivity_macro_12             0.1055  ↑
  specificity_macro_12             0.9244  ↑
  precision_macro_12               0.2973  ↑
  recall_macro_12                  0.1055  ↑
  hamming_loss_12                  0.2748  ↓
  jaccard_12                       0.0842  ↑
  label_cooc_consistency_12        0.1212  ↓


# Summary: CC vs ECC vs BR (RF baseline)

BR+RF results are pasted from baseline.ipynb for direct comparison.

In [6]:
# BR+RF macro results from baseline.ipynb
# (paste updated values here once baseline is re-evaluated with evaluate_12)
br_rf_macro = {
    'model'                    : 'BR_RF',
    'roc_auc_12'               : 0.820,
    'pr_auc_12'                : 0.624,
    'f1_macro_12'              : 0.546,
    'instance_f1_12'           : None,   # fill in from baseline
    'balanced_accuracy_12'     : 0.689,
    'matched_accuracy_12'      : None,   # fill in from baseline
    'sensitivity_macro_12'     : 0.487,
    'specificity_macro_12'     : 0.892,
    'precision_macro_12'       : None,   # fill in from baseline
    'recall_macro_12'          : 0.487,
    'hamming_loss_12'          : None,   # fill in from baseline
    'jaccard_12'               : None,   # fill in from baseline
    'label_cooc_consistency_12': None,   # fill in from baseline
}

METRIC_KEYS = [
    'roc_auc_12', 'pr_auc_12', 'f1_macro_12', 'instance_f1_12',
    'balanced_accuracy_12', 'matched_accuracy_12',
    'sensitivity_macro_12', 'specificity_macro_12',
    'precision_macro_12', 'recall_macro_12',
    'hamming_loss_12', 'jaccard_12', 'label_cooc_consistency_12',
]

summary_df = (
    pd.DataFrame([br_rf_macro, metrics_cc, metrics_ecc])
    .set_index('model')[METRIC_KEYS]
)

print('Macro-averaged test metrics: BR vs CC vs ECC (RF base)')
print(summary_df.to_string())


Macro-averaged test metrics: BR vs CC vs ECC (RF base)
        roc_auc_12  pr_auc_12  f1_macro_12  instance_f1_12  balanced_accuracy_12  matched_accuracy_12  sensitivity_macro_12  specificity_macro_12  precision_macro_12  recall_macro_12  hamming_loss_12  jaccard_12  label_cooc_consistency_12
model                                                                                                                                                                                                                                         
BR_RF       0.8200     0.6240       0.5460             NaN                0.6890                  NaN                0.4870                0.8920                 NaN           0.4870              NaN         NaN                        NaN
CC_RF       0.4879     0.2601       0.1887          0.1697                0.5025               0.0037                0.2145                0.7904              0.2509           0.2145           0.3597      0.1096                 